# 34_alarm_chatbot_JIW

- 목적: 경보 근거 카드(32)와 결과 요약(28·33)을 **대화로 묻고 답하는 챗봇 인터페이스**를 만들고, 답이 근거 안에 머무르는지 점검한다. 설계: `docs/chatbot_design_JIW.md`.
- 입력: `results/28_model_ensemble_JIW`, `results/32_alarm_explain_JIW/cards.csv`, `results/33_early_warning_JIW`의 CSV. **모델 학습·가중치 불필요.**
- 출력: `results/34_alarm_chatbot_JIW/`(facts.csv · eval_offline.csv · dialogue.csv · guardrail_test.csv)
- 담당: JIW
- 탐지 대상: 유압펌프 모터-펌프 구동부 이상 (센서 부착 부위 미확정이라 원인을 단정하지 않는다)
- 심사 대응: 4번(경보 → 조치 설명 인터페이스), 5번(근거 기반 LLM 적용)
- 근거 표기: **[데이터]** 이 노트북 출력 / **[가정]** 운영 전제

## 먼저 한계

- 챗봇은 **판정하지 않는다.** 경보 여부는 28번 규칙(코드)이 정하고, 챗봇은 그 근거를 풀어 설명한다. 그래서 챗봇 오류가 탐지 성능(AUC·오경보율)에 영향을 주지 않는다.
- 이 노트북은 **오프라인 백엔드**(키워드 라우팅 + 템플릿)만 실행한다. 외부 LLM 백엔드는 API 키가 필요하고 대회 규정상 외부 호출 가능 여부를 확인하지 않아 **실제 호출로 시험하지 않았다.** §4는 가짜 클라이언트로 요청 형태와 안전장치만 점검한다.
- 질문 세트 14개는 내가 만든 회귀 점검용이다. 독립적인 성능 평가가 아니다.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "34_alarm_chatbot_JIW"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)
import chatbot_jiw as C
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 200)
print(NB, RES.relative_to(ROOT))

34_alarm_chatbot_JIW results\34_alarm_chatbot_JIW


## 1. 구조

```
질문 ─▶ [백엔드] ─▶ 답 ─▶ [숫자 검증] [단정 표현 검사] ─▶ 경고 표시
          │
          ├ 오프라인: 키워드 → 주제 → facts 문장 인용 (결정적, 기본)
          └ LLM(선택): system = 규칙 + FACTS + 선택한 카드, 대화 이력 포함

FACTS = 결과 CSV에서 읽은 수치 문장 17개(F1~F17). 모든 답은 [F#]로 인용한다.
CARD  = 32번 근거 카드 문장(선택한 경보 1건)
```

안전장치(코드로 검사하며 LLM 백엔드에도 똑같이 적용된다)
1. **숫자 검증**: 답의 숫자가 FACTS·CARD 텍스트에 없으면 경고.
2. **단정 표현 검사**: "모터/펌프/베어링 … 고장입니다", "N분 후 경보" 같은 원인·시점 단정을 정규식으로 찾아 경고(휴리스틱이라 놓칠 수 있다).
3. **범위 밖 질문**은 "제공된 근거에 없습니다."로 답한다.
4. 외부 호출 실패 시 오프라인 답으로 대체한다.

In [2]:
facts = C.build_facts()
cards = C.load_cards()
ft = pd.Series(facts, name="내용").rename_axis("번호").to_frame()
ft.to_csv(RES / "facts.csv")
print(len(facts), "개 facts, 카드", len(cards), "건 (실제 이상 창", int((cards.label == 1).sum()), "/ 정상 오경보 창", int((cards.label == 0).sum()), ")")
ft

17 개 facts, 카드 64 건 (실제 이상 창 63 / 정상 오경보 창 1 )


,내용
번호,
F1,"경보 규칙: 주의 = CNN 점수가 임계를 넘음, 경보 = CNN과 MCD 점수가 모두 임계를 넘음. 경보는 항상 주의의 부분집합이다. CNN은 신호의 다음 값을 예측해 틀린 정도를 보고, MCD는 윈도우 통계 18개가 정상 무리에서 멀리 떨어진 정도를 본다."
F2,임계는 학습용 정상 점수의 상위 1%(q99)로만 정하고 이상 데이터로 조정하지 않는다. 점수는 임계로 나눈 배수로 표시한다(1배가 임계).
F3,"CNN 단독(주의 단계) 세그먼트 오경보율은 2.8% / 1.8%(group_kfold / time_block), 시간당 약 12.9 / 8.1건이다. 실제 이상 세그먼트는 전부 탐지했다."
F4,"CNN과 MCD를 모두 넘어야 하는 경보 단계의 세그먼트 오경보율은 0.2% / 0.1%, 시간당 약 1.0 / 0.4건이다. 실제 이상 세그먼트는 전부 탐지했다(미탐 0건)."
F5,"MCD 단독 세그먼트 오경보율은 3.8% / 3.1%, CNN 또는 MCD(OR)는 6.4% / 4.8%로 오경보가 늘어 채택하지 않았다."
F6,"약한 이상 탐지: 합성 강도 2에서 경보(AND)의 탐지율은 CNN 단독 대비 32% / 44%(group_kfold / time_block)만 유지되고, 강도 8에서는 87% / 94%다. 합성 강도 1 이하의 약한 이상은 경보도 주의도 거의 잡지 못한다(CNN 단독 강도 1 탐지율 33.6% / 31.5%, 경보 2.3% / 3.0%)."
F7,"오경보는 고부하 운전 구간에 몰린다. CNN 단독 고부하 오경보 3.2% / 2.2%, 저부하 0.1% / 0.0%(group_kfold / time_block). 운전 상태별 임계로 바꿔도 고부하 오경보는 3.1% / 3.0%로 줄지 않아 채택하지 않았다. 경보 단계(CNN AND MCD)가 이 오경보를 지운다."
F8,"조기경보(합성 열화 시나리오): 열화가 완성되기 전에 알린 선행 시간 중앙값은 주의 1.39 / 3.60 / 7.39분, 경보 0.98 / 2.81 / 6.71분이다(열화 2 / 5 / 10분 기준). 선행 시간은 열화 기간의 약 70%라서 실제 설비의 열화 속도를 모르면 '몇 분 전'을 말할 수 없다."
F9,"정상 구간에서 거짓 사건은 주의 시간당 6.2건, 경보 시간당 0.3건이다(정상 약 3.1시간 분량, 사건 수가 적어 불확실하다)."


## 2. 질문 세트 점검 (오프라인 백엔드)

14개 질문: 답해야 하는 10개(필수 인용 확인), 단정을 거절해야 하는 2개(시점 예측, 원인 단정), 범위 밖 2개.

In [3]:
ev = C.evaluate(C.OfflineBackend(), facts, cards)
ev.to_csv(RES / "eval_offline.csv", index=False)
print("통과", int(ev["통과"].sum()), "/", len(ev), " 근거 없는 숫자 합", int(ev["근거 없는 숫자"].sum()), " 단정 표현 합", int(ev["단정 표현"].sum()))
ev

통과 14 / 14  근거 없는 숫자 합 0  단정 표현 합 0


,질문,종류,인용,필수 인용 충족,거절,근거 없는 숫자,단정 표현,통과
0,경보랑 주의는 뭐가 달라?,answer,"F1,F2,F3,F4",True,False,0,0,True
1,임계값은 어떻게 정해?,answer,"F1,F2,F3,F4",True,False,0,0,True
2,오경보는 얼마나 자주 나?,answer,"F3,F4,F7,F9",True,False,0,0,True
3,고부하에서 오경보가 왜 많아?,answer,"F3,F4,F7,F9",True,False,0,0,True
4,약한 이상도 잡아?,answer,"F6,F16,F17",True,False,0,0,True
5,조기경보는 몇 분 전에 울려?,answer,"F8,F11,F16",True,False,0,0,True
6,미탐은 어떤 경우야?,answer,"F6,F16,F17",True,False,0,0,True
7,어떻게 점검해야 해?,answer,"F14,F15",True,False,0,0,True
8,이 결과를 얼마나 믿을 수 있어?,answer,"F12,F15,F16",True,False,0,0,True
9,CNN 단독이랑 MCD 단독 비교해줘,answer,"F3,F4,F5",True,False,0,0,True


**발견 [데이터]** 14개 모두 통과한다. 이는 **키워드 라우터가 내가 쓴 질문 표현을 알고 있다는 뜻**이어서 일반적인 대화 성능의 증거가 아니다. 이 점검이 보장하는 것은 (1) 답이 facts 문장 그대로라 숫자가 어긋나지 않는다는 것, (2) 시점 예측·원인 질문에는 거절 근거(F11·F15)를 붙여 답한다는 것, (3) 범위 밖 질문에 "근거에 없습니다"로 답한다는 것뿐이다. 표현이 다른 질문은 라우팅에 실패해 "근거에 없습니다"로 떨어질 수 있다. 자유로운 질문은 LLM 백엔드의 몫이다.

## 3. 대화 예시: 경보 카드 선택

실제 이상 창 카드 하나와 AND 오경보 카드 하나를 골라 같은 질문을 한다.

In [4]:
rows = []
for label, idx in [("실제 이상 창", int(cards.index[cards.label == 1][0])), ("AND 오경보 창", int(cards.index[cards.label == 0][0]))]:
    s = C.ChatSession(C.OfflineBackend(), facts, cards)
    s.select_card(idx)
    for q in ["이 경보 왜 울렸어?", "어떻게 점검해야 해?", "고장 원인이 모터 베어링이야?", "몇 분 후에 경보가 울릴지 예측해줘"]:
        r = s.ask(q)
        rows.append({"카드": label, "질문": q, "답": r["answer"], "인용": ",".join(r["cited"]), "거절": r["refused"]})
dlg = pd.DataFrame(rows)
dlg.to_csv(RES / "dialogue.csv", index=False)
for _, r in dlg.iterrows():
    print("[" + r["카드"] + "] Q. " + r["질문"] + "\nA. " + r["답"] + "\n")

[실제 이상 창] Q. 이 경보 왜 울렸어?
A. 선택한 경보의 근거 카드입니다.
[경보] 상부 진동 쪽 이상 의심. CNN 점수가 임계의 33.5배, MCD 점수가 임계의 2.1배. 예측 오차 비중: 상부 진동 69% · 하부 진동 14% · 전류 16%. 정상에서 가장 벗어난 통계: 상부 진동 RMS +22.1σ, 상부 진동 피크-피크 +20.7σ, 상부 진동 피크 +17.8σ. 운전 상태 추정: 저부하. 점검 권고(가정): 상단 체결부·커플링·정렬.
예측 오차가 몰린 신호는 '상부 진동'이지만 이것이 원인 부위라는 뜻은 아닙니다 [F13][F15]. 점검 권고는 가정입니다 [F14].

[실제 이상 창] Q. 어떻게 점검해야 해?
A. 근거 요약입니다.
- 점검 권고(가정): 상부 진동 우세 → 상단 체결부·커플링·정렬, 하부 진동 우세 → 하단 베이스·마운트, 전류 우세 → 전원·부하 변동·토출 압력, 진동과 전류 동시 → 구동부 전반 종합 점검. 가이드북과 일반 설비 상식에서 온 가정이며 이 데이터로 검증하지 않았다. [F14]
- 센서 부착 부위(모터 몸체인지 펌프 몸체인지)를 데이터로 확정할 수 없어 원인이 모터인지 펌프인지, 어떤 부품인지는 구분할 수 없다. 탐지 대상은 '유압펌프 모터-펌프 구동부 이상'으로 쓴다. [F15]

[실제 이상 창] Q. 고장 원인이 모터 베어링이야?
A. 원인(모터/펌프/부품)은 이 데이터로 구분할 수 없습니다. 근거로 말할 수 있는 것은 아래입니다.
- 센서 부착 부위(모터 몸체인지 펌프 몸체인지)를 데이터로 확정할 수 없어 원인이 모터인지 펌프인지, 어떤 부품인지는 구분할 수 없다. 탐지 대상은 '유압펌프 모터-펌프 구동부 이상'으로 쓴다. [F15]
- 근거 카드 정합성: 합성 이상에서 CNN 예측 오차 비중 1위 채널이 주입한 채널과 일치한 비율은 강도 2 이상의 유형·강도별 칸에서 94~100%다(합성 검증이며 실제 원인 진단 성능이 아니다). [F12]
- 실제 이상 창 63개의 오차 비중 1위 채널은 

**발견 [데이터]** 같은 질문에 카드가 바뀌면 답의 첫 부분(카드 문장)만 바뀌고, 근거·한계 설명은 같다. 오경보 카드는 CNN 배수 1.0 · MCD 배수 1.7로 임계 바로 위라는 점이 문장에 그대로 드러난다. 원인을 묻는 질문에는 "구분할 수 없다"와 근거(F12·F13·F15)가, 시점 예측 질문에는 쓰지 않는 이유(F11)가 따라온다.

## 4. 안전장치와 LLM 백엔드 요청 형태 점검 (가짜 클라이언트)

실제 외부 호출 없이 **LLM이 규칙을 어기는 답을 냈을 때** 숫자 검증과 단정 표현 검사가 잡는지, 그리고 LLM 백엔드가 어떤 요청을 만드는지 본다. 가짜 클라이언트가 미리 정한 답을 돌려준다.

In [5]:
class _Block: type = "text"
class _Resp:
    stop_reason = "end_turn"
    def __init__(self, text): b = _Block(); b.text = text; self.content = [b]
class _Beta:
    def __init__(self, reply): self.reply, self.calls = reply, []
    def create(self, **kw): self.calls.append(kw); return _Resp(self.reply)
class _Fake:
    def __init__(self, reply): self.beta = type("N", (), {"messages": _Beta(reply)})()

CASES = [("정상 답(근거 안)", "경보 단계 오경보는 시간당 1.0 / 0.4건입니다 [F4]"),
         ("없는 숫자를 만든 답", "오경보는 시간당 99.9건입니다 [F3]"),
         ("원인을 단정한 답", "이 경보는 모터 베어링 고장입니다 [F13]"),
         ("시점을 예측한 답", "약 3분 후 경보 수준에 도달합니다 [F8]")]
rows = []
for name, reply in CASES:
    fake = _Fake(reply)
    r = C.ChatSession(C.LLMBackend(client=fake), facts, cards).ask("오경보 얼마나 나?")
    rows.append({"사례": name, "근거 없는 숫자": ",".join(r["unsupported_numbers"]), "단정 표현": ",".join(r["unsafe_claims"]),
                 "경고 표시됨": "⚠" in r["answer"]})
gt = pd.DataFrame(rows)
gt.to_csv(RES / "guardrail_test.csv", index=False)
req = fake.beta.messages.calls[0]
print("LLM 백엔드가 만든 요청 (system은 앞 80자만):", {k: (v[:80] + "...") if k == "system" else v for k, v in req.items() if k != "messages"})
gt

LLM 백엔드가 만든 요청 (system은 앞 80자만): {'model': 'claude-opus-5-5', 'max_tokens': 4000, 'system': '당신은 프레스 유압펌프 이상탐지 결과를 설명하는 챗봇입니다. 아래 [FACTS]와 [CARD]만 근거로 한국어로 답합니다.\n규칙:\n1. 답에 쓰...', 'output_config': {'effort': 'low'}, 'betas': ['server-side-fallback-2026-07-01'], 'fallbacks': 'default'}


,사례,근거 없는 숫자,단정 표현,경고 표시됨
0,정상 답(근거 안),,,False
1,없는 숫자를 만든 답,99.9,,True
2,원인을 단정한 답,,모터 베어링 고장입니다,True
3,시점을 예측한 답,,3분 후 경보 수준에 도달,True


**발견 [데이터]**
- 근거 안의 답은 경고 없이 통과하고, 없는 숫자(99.9)·원인 단정·시점 예측은 모두 경고가 붙는다(위 표). 단정 표현 검사는 정규식 휴리스틱이라 다른 표현은 놓칠 수 있다.
- 요청에는 facts 요약과 대화 이력만 들어가고 원 데이터·윈도우 점수는 들어가지 않는다. 모델은 환경 변수로 바꾼다(기본값은 코드 참고).
- **시험하지 않은 것**: 실제 LLM 응답의 품질, 환각 빈도, 응답 지연·비용. 가짜 클라이언트로는 알 수 없다. 키가 있는 환경에서 `python src/chatbot_jiw.py --llm`으로 시험하고 §2의 질문 세트를 같은 방식으로 돌려 비교해야 한다.

## 5. 해석

발견 → 시사점 → 활용은 `reports/34_alarm_chatbot_JIW.md`에 정리했다.